# 19. tSZ angular power spectrum across lightcone shells

Measure $C_\ell$ of the FLAMINGO L2p8_m9 lightcone-0 Compton-$y$ maps for the
**nearest five shells** (shell 0–4, $z\lesssim0.25$, NSIDE=4096) and compare each to a
halo-model tSZ theory whose pressure-profile amplitude is built **from the nb17
$Y$–$M$ fit** (no fit to the power spectrum). This shows how the tSZ spectrum builds up
with redshift, shell by shell.

Pipeline (nb05 convention): subtract monopole, `anafast`, deconvolve the pixel window,
$/f_\mathrm{sky}$, $D_\ell=\ell(\ell+1)C_\ell/2\pi$.

**Theory:** `SelfSimilarGNFWPressureProfile` with $P_{500}=P_{500,0}(M_{500c}/M_\odot)^{\alpha_P}
E(z)^{2+\beta_P}$, the GNFW **shape fixed to the FLAMINGO illustrative profile**
($c_{500}=1.77,\alpha=1.48,\beta=4.55,\gamma=0.31$ — close to Arnaud+10). Amplitude from
nb17: pressure mass slope $\alpha_P=\alpha_{YM}-1$ (the $r_{500c}^3\propto M$ factor),
$\beta_P=\beta_{YM}$, and $P_{500,0}$ from the nb17 intercept. Each shell's theory uses
the halo-model Limber integral restricted to that shell's redshift range.

In [1]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import sys
sys.path.insert(0, "/scratch/scratch-lxu/flamingo_repo/src")

import numpy as np
import healpy as hp
import matplotlib.pyplot as plt

# Publication-quality plot defaults
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "axes.linewidth": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
})

import jax.numpy as jnp

from flamingo.catalogue import D3A_COSMOLOGY
from flamingo.profiles.custom_pressure import SelfSimilarGNFWPressureProfile
from hmfast.halos import HaloModel
from hmfast.tracers import tSZTracer
from hmfast.utils import Const

MAP_TMPL = ("/rds/rds-lxu/flamingo/L2p8_m9/lightcone0/healpix_map/snapshots/"
            "comptonY_L2p8_m9_lc0_nside4096_shell{idx}.fits")
# shell index -> (z_lo, z_hi) from the catalogue shell_idx ranges
SHELLS = [(0, 0.0032, 0.050), (1, 0.050, 0.100), (2, 0.100, 0.150),
          (3, 0.150, 0.200), (4, 0.200, 0.250)]
LMAX = 6000
M_PIV = 6.0e14
# nb17 lowest-z Y-M fit (Y = D_A^2 Y in Mpc^2)
ALPHA_YM, BETA_YM, C17 = 1.592, 0.526, -9.266
# FLAMINGO illustrative GNFW shape (gamma = 0.31)
SHAPE = dict(P0=6.05, c500=1.77, alpha=1.48, beta=4.55, gamma=0.31)


## Amplitude $P_{500,0}$ from the nb17 intercept

$Y_{5R500c}=\frac{\sigma_T}{m_ec^2}P_{500}P_0\,r_{500c}^3 I_\mathrm{shape}$, and since
$r_{500c}^3\propto M$, the pressure mass slope is $\alpha_P=\alpha_{YM}-1$ (using $\alpha_{YM}$
directly would double-count the mass scaling). $P_{500,0}$ is set so
$Y_{5R500c}(M_\mathrm{piv},z{=}0)=e^{C}$.

In [2]:
h = D3A_COSMOLOGY.H0 / 100.0
rho_crit0 = 2.77536627e11 * h ** 2
r500c_Mpc = (3 * M_PIV / (4 * np.pi * 500 * rho_crit0)) ** (1.0 / 3.0)
mpc_cm = Const._Mpc_over_m_ * 100.0
sigma_T_cm2, m_e_c2_eV = 6.6524587e-25, 510998.95
ALPHA_P, BETA_P = ALPHA_YM - 1.0, BETA_YM

xg = np.logspace(-4, np.log10(5.0), 5000)
p_shape = (SHAPE['c500'] * xg) ** (-SHAPE['gamma']) * (
    1 + (SHAPE['c500'] * xg) ** SHAPE['alpha']) ** ((SHAPE['gamma'] - SHAPE['beta']) / SHAPE['alpha'])
I_shape = np.trapezoid(p_shape * 4 * np.pi * xg ** 2, xg)
Y_unit = (sigma_T_cm2 / m_e_c2_eV) * (M_PIV ** ALPHA_P) * SHAPE['P0'] \
         * (r500c_Mpc * mpc_cm) ** 3 * I_shape / mpc_cm ** 2
P500_0 = np.exp(C17) / Y_unit
print(f"alpha_P={ALPHA_P:.3f}  I_shape={I_shape:.3f}  P500_0={P500_0:.3e}")

# profile = SelfSimilarGNFWPressureProfile(
#     P500_0=P500_0, alpha_amp=ALPHA_P, beta_amp=BETA_P, B=1.0, **SHAPE)
profile = SelfSimilarGNFWPressureProfile(
    P500_0=P500_0, alpha_amp=ALPHA_P, beta_amp=BETA_P, B=1.1, **SHAPE)

alpha_P=0.592  I_shape=0.742  P500_0=4.206e-09


In [3]:
def measure_dl(path, lmax=LMAX):
    m = hp.read_map(path, dtype=np.float64)
    nside = hp.npix2nside(m.size)
    fsky = float(np.mean(m != 0.0))
    cl = hp.anafast(m - m.mean(), lmax=lmax, iter=0)
    ell = np.arange(cl.size)
    cl = cl / (hp.pixwin(nside, lmax=lmax) ** 2 * fsky)
    return ell, ell * (ell + 1) / (2 * np.pi) * cl

def log_bin(ell, dl, lmin=10, lmax=LMAX, nbin=24):
    edges = np.logspace(np.log10(lmin), np.log10(lmax), nbin + 1)
    idx = np.digitize(ell, edges) - 1
    lb, db = [], []
    for b in range(nbin):
        s = (idx == b) & (ell >= lmin)
        if s.any():
            lb.append(ell[s].mean()); db.append(dl[s].mean())
    return np.array(lb), np.array(db)

hm = HaloModel(cosmology=D3A_COSMOLOGY)
ell_th = jnp.logspace(1.0, np.log10(LMAX), 40)
m_grid = jnp.logspace(11.0, 15.5, 50)
pref = np.asarray(ell_th) * (np.asarray(ell_th) + 1) / (2 * np.pi)

def theory_dl(zlo, zhi):
    z_grid = jnp.linspace(zlo, zhi, 30)
    tr = tSZTracer(profile=profile)
    return pref * (np.asarray(hm.cl_1h(tr, tr, l=ell_th, m=m_grid, z=z_grid))
                   + np.asarray(hm.cl_2h(tr, tr, l=ell_th, m=m_grid, z=z_grid)))

I0000 00:00:1782460267.849282 2559753 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782460267.849650 2559753 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1782460268.840898 2559753 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782460268.841167 2559753 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## tSZ power spectrum, shell by shell

In [4]:
results = []
for idx, zlo, zhi in SHELLS:
    ell, dl = measure_dl(MAP_TMPL.format(idx=idx))
    ellb, dlb = log_bin(ell, dl)
    dth = theory_dl(zlo, zhi)
    results.append(dict(idx=idx, zlo=zlo, zhi=zhi, ellb=ellb, dlb=dlb, dth=dth))
    print(f"shell {idx} (z {zlo:.2f}-{zhi:.2f}): measured D_3000={np.interp(3000, ell, dl):.2e}  "
          f"theory D_3000={np.interp(3000, np.asarray(ell_th), dth):.2e}")

colors = plt.cm.viridis(np.linspace(0.1, 0.85, len(SHELLS)))
fig, ax = plt.subplots(figsize=(8, 6))
for r, col in zip(results, colors):
    zc = 0.5 * (r['zlo'] + r['zhi'])
    ax.plot(r['ellb'], r['dlb'], 'o', ms=4, color=col,
            label=fr"shell {r['idx']} ($z\simeq${zc:.2f})")
    ax.plot(np.asarray(ell_th), r['dth'], '-', lw=1.8, color=col)
ax.plot([], [], 'k-', label='theory (nb17 amp + FLAMINGO GNFW)')
ax.plot([], [], 'ko', label='measured')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(r'$\ell$'); ax.set_ylabel(r'$D_\ell=\ell(\ell+1)C_\ell/2\pi$')
ax.set_xlim(20, LMAX); ax.set_ylim(1e-16, 1e-12)
ax.set_title('tSZ power spectrum per lightcone shell (points=map, lines=theory)')
ax.legend(fontsize=8, ncol=2)
fig.tight_layout(); plt.show()

shell 0 (z 0.00-0.05): measured D_3000=4.66e-16  theory D_3000=9.98e-16


shell 1 (z 0.05-0.10): measured D_3000=1.08e-14  theory D_3000=1.42e-14


shell 2 (z 0.10-0.15): measured D_3000=2.99e-14  theory D_3000=4.13e-14


shell 3 (z 0.15-0.20): measured D_3000=5.58e-14  theory D_3000=6.73e-14


shell 4 (z 0.20-0.25): measured D_3000=7.32e-14  theory D_3000=8.54e-14


In [5]:
# Cumulative spectrum: shells are independent, so the z<0.25 total is the sum.
ell_full, _ = measure_dl(MAP_TMPL.format(idx=0))
dl_sum = np.zeros_like(results[0]['dlb'])
for r in results:
    dl_sum = dl_sum + r['dlb']
dth_sum = np.sum([r['dth'] for r in results], axis=0)

fig, ax = plt.subplots(figsize=(7.6, 5.6))
for r, col in zip(results, colors):
    ax.plot(r['ellb'], r['dlb'], '-', lw=0.8, color=col, alpha=0.6)
ax.plot(results[0]['ellb'], dl_sum, 'o-', color='k', ms=4, lw=1.5,
        label=r'measured $\sum$ shells 0-4 ($z<0.25$)')
ax.plot(np.asarray(ell_th), dth_sum, 'r-', lw=2,
        label=r'theory $\sum$ shells 0-4')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(r'$\ell$'); ax.set_ylabel(r'$D_\ell$')
ax.set_xlim(20, LMAX); ax.set_ylim(1e-15, 3e-12)
ax.set_title(r'Cumulative tSZ power spectrum, $z<0.25$ (thin = individual shells)')
ax.legend(fontsize=9); fig.tight_layout(); plt.show()

### Notes

- Each successive shell (higher $z$) adds power and shifts its peak to **higher $\ell$**:
  more distant structure subtends smaller angles. This is how the full tSZ spectrum
  (peak $\ell\sim3000$) is assembled from the redshift shells.
- The theory amplitude is **derived from the nb17 $Y$–$M$ fit** (with the $\alpha_P=
  \alpha_{YM}-1$ pressure-slope shift) and the FLAMINGO illustrative GNFW shape
  ($\gamma=0.31$); each shell's curve uses the same profile with the Limber integral
  restricted to that shell — no per-shell tuning.
- Shells are largely uncorrelated, so the $z<0.25$ total is well approximated by the
  sum of shells; only the nearest five shells are on disk, so this is a partial sum,
  not the full lightcone.